In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import fbeta_score
from lightgbm import LGBMClassifier

In [5]:
# 파일 경로: 상황에 맞게 바꿔
TRAIN_PATH_CSV = "../data/train.csv"
TEST_PATH_CSV = "../data/test.csv"
TRAIN_PATH_PARQUET = "/mnt/data/train.parquet"
TEST_PATH_PARQUET = "/mnt/data/test.parquet"

SUBMIT_DIR = "../rslt/08"
os.makedirs(SUBMIT_DIR, exist_ok=True)


# =========================================================
# 1. 데이터 로드
# =========================================================
def load_data():
    if os.path.exists(TRAIN_PATH_CSV) and os.path.exists(TEST_PATH_CSV):
        train = pd.read_csv(TRAIN_PATH_CSV)
        test = pd.read_csv(TEST_PATH_CSV)
    elif os.path.exists(TRAIN_PATH_PARQUET) and os.path.exists(TEST_PATH_PARQUET):
        train = pd.read_parquet(TRAIN_PATH_PARQUET)
        test = pd.read_parquet(TEST_PATH_PARQUET)
    else:
        raise FileNotFoundError("train/test 파일 경로를 확인해줘.")

    train.columns = train.columns.str.strip()
    test.columns = test.columns.str.strip()

    return train, test


train, test = load_data()

In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# =========================================================
# 1. 사용할 feature
# =========================================================
selected_features = [
    "ROA(C) before interest and depreciation before interest",
    "ROA(A) before interest and % after tax",
    "ROA(B) before interest and depreciation after tax",
    "Persistent EPS in the Last Four Seasons",
    "Per Share Net profit before tax (Yuan ¥)",
    "Operating Gross Margin",
    "Operating Profit Per Share (Yuan ¥)",
    "Operating Profit Rate",
    "Net Value Per Share (A)",
    "Net Value Per Share (B)",
    "Net Value Per Share (C)",
    "Current Liability to Assets",
    "Working Capital to Total Assets",
    "Quick Assets/Total Assets",
    "Current Assets/Total Assets",
    "Cash/Total Assets",
    "Operating profit/Paid-in capital",
    "Net profit before tax/Paid-in capital",
    "Total Asset Turnover",
    "Inventory Turnover Rate (times)",
    "Net Income to Total Assets",
    "Net worth/Assets",
    "Debt ratio %",
    "Cash Flow to Total Assets",
    "CFO to Assets",
    "Cash Flow to Liability",
    "Cash Flow to Equity",
    "Retained Earnings to Total Assets",
    "Total expense/Assets",
    "Gross Profit to Sales",
    "Equity to Liability",
    "Operating Funds to Liability",
    "Current Liability to Liability",
    "Borrowing dependency"
]

# =========================================================
# 2. 데이터 준비
# train dataframe이 있다고 가정
# target = 'Bankrupt?'
# =========================================================
X = train[selected_features].copy()
y = train["Bankrupt?"].copy()

# 혹시 결측치 있으면 중앙값으로 채우기
for col in X.columns:
    X[col] = X[col].fillna(X[col].median())

# =========================================================
# 3. train/valid split
# =========================================================
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================================================
# 4. clipping 함수
# train 기준 quantile로 자르고
# val에도 같은 기준 적용
# =========================================================
def clip_by_train_quantile(X_train, X_val, cols, lower_q=0.01, upper_q=0.99):
    X_train_clipped = X_train.copy()
    X_val_clipped = X_val.copy()
    clip_dict = {}

    for col in cols:
        lower = X_train[col].quantile(lower_q)
        upper = X_train[col].quantile(upper_q)
        clip_dict[col] = (lower, upper)

        X_train_clipped[col] = X_train_clipped[col].clip(lower, upper)
        X_val_clipped[col] = X_val_clipped[col].clip(lower, upper)

    return X_train_clipped, X_val_clipped, clip_dict

X_train_clip, X_val_clip, clip_info = clip_by_train_quantile(
    X_train, X_val, selected_features, lower_q=0.01, upper_q=0.99
)

# =========================================================
# 5. SMOTE
# =========================================================
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_clip, y_train)

print("원래 train shape:", X_train_clip.shape, y_train.shape)
print("SMOTE 후 shape:", X_train_sm.shape, y_train_sm.shape)
print("원래 class 비율:")
print(y_train.value_counts())
print("SMOTE 후 class 비율:")
print(pd.Series(y_train_sm).value_counts())

# =========================================================
# 6. XGBoost 모델
# =========================================================
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train_sm, y_train_sm)

# =========================================================
# 7. validation 예측 확률
# =========================================================
val_proba = model.predict_proba(X_val_clip)[:, 1]

# =========================================================
# 8. threshold 최적화 (F2 기준)
# =========================================================
thresholds = np.arange(0.01, 0.51, 0.01)

best_f2 = 0
best_threshold = 0.5

results = []

for th in thresholds:
    val_pred = (val_proba >= th).astype(int)
    f2 = fbeta_score(y_val, val_pred, beta=2)

    results.append([th, f2])

    if f2 > best_f2:
        best_f2 = f2
        best_threshold = th

results_df = pd.DataFrame(results, columns=["threshold", "f2_score"])
print(results_df.sort_values("f2_score", ascending=False).head(10))

print("\n최적 threshold:", best_threshold)
print("최고 F2-score:", best_f2)

# =========================================================
# 9. 최적 threshold 기준 최종 평가
# =========================================================
best_pred = (val_proba >= best_threshold).astype(int)

print("\n[Confusion Matrix]")
print(confusion_matrix(y_val, best_pred))

print("\n[Classification Report]")
print(classification_report(y_val, best_pred, digits=4))

print("\n[Final Validation F2]")
print(fbeta_score(y_val, best_pred, beta=2))

원래 train shape: (3818, 34) (3818,)
SMOTE 후 shape: (7390, 34) (7390,)
원래 class 비율:
Bankrupt?
0    3695
1     123
Name: count, dtype: int64
SMOTE 후 class 비율:
Bankrupt?
0    3695
1    3695
Name: count, dtype: int64
    threshold  f2_score
31       0.32  0.555556
33       0.34  0.555556
32       0.33  0.555556
30       0.31  0.552764
29       0.30  0.547264
37       0.38  0.544041
36       0.37  0.544041
16       0.17  0.542986
35       0.36  0.541237
15       0.16  0.540541

최적 threshold: 0.32
최고 F2-score: 0.5555555555555556

[Confusion Matrix]
[[872  52]
 [  9  22]]

[Classification Report]
              precision    recall  f1-score   support

           0     0.9898    0.9437    0.9662       924
           1     0.2973    0.7097    0.4190        31

    accuracy                         0.9361       955
   macro avg     0.6435    0.8267    0.6926       955
weighted avg     0.9673    0.9361    0.9484       955


[Final Validation F2]
0.5555555555555556


In [9]:
# =========================================================
# 10. test 데이터 전처리
# =========================================================
X_test = test[selected_features].copy()

for col in X_test.columns:
    X_test[col] = X_test[col].fillna(X_train[col].median())

# train 기준 clipping 적용
for col in selected_features:
    lower, upper = clip_info[col]
    X_test[col] = X_test[col].clip(lower, upper)

# =========================================================
# 11. test 예측
# =========================================================
test_proba = model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

# =========================================================
# 12. 제출 파일 저장
# sample_submission 형식 없으면 일단 이렇게
# =========================================================
submission = pd.DataFrame({
    "ID": test["ID"],
    "Bankrupt?": test_pred
})

SUBMIT_DIR = "../rslt/10"
os.makedirs(SUBMIT_DIR, exist_ok=True)

submission.to_csv("../rslt/10/result.csv", index=False)
print("저장 완료")
print(submission.head())

저장 완료
     ID  Bankrupt?
0  4773          0
1  4774          0
2  4775          0
3  4776          0
4  4777          0
